In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load the cleaned data
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

# Rename columns to make them formula-safe
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "pct_highschool_or_more_(2023)": "pct_hs_2023",
    "Pop 2023": "pop_2023"
})

# Convert to numeric where needed
df['church_density_1890'] = pd.to_numeric(df['church_density_1890'], errors='coerce')
df['pop_2023'] = pd.to_numeric(df['pop_2023'], errors='coerce')

# Drop rows with missing key data
df = df.dropna(subset=['church_density_1890', 'pct_hs_2023', 'income_1989_real_2023', 'pct_hs_1990', 'pop_2023'])

# Log transform population
df['log_pop_2023'] = np.log(df['pop_2023'])

# Standardize controls
scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real_2023', 'pct_hs_1990']]
)

# Run regression
model = smf.ols(
    formula='pct_hs_2023 ~ church_density_1890 + income_1989_scaled + pct_hs_1990_scaled + log_pop_2023 + C(State)',
    data=df
).fit(cov_type='HC3')

# Display results
print(model.summary())



                            OLS Regression Results                            
Dep. Variable:            pct_hs_2023   R-squared:                       0.668
Model:                            OLS   Adj. R-squared:                  0.662
Method:                 Least Squares   F-statistic:                     104.9
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        11:28:39   Log-Likelihood:                -6684.8
No. Observations:                2659   AIC:                         1.347e+04
Df Residuals:                    2608   BIC:                         1.377e+04
Df Model:                          50                                         
Covariance Type:                  HC3                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              94.6585    